In [16]:
# General Python imports
import os, sys
from dotenv import load_dotenv
from pathlib import Path
import numpy as np

# NANOGRAV imports
import pint
from pint.models import get_model_and_toas
from pint.residuals import Residuals
from pint.simulation import zero_residuals
pint.logging.setup(sink=sys.stderr, level="WARNING", usecolors=True)

2

In [17]:
# PARAMS TO RECOVER
# Amplitude of red noise in GW units 
# [-18, -13]
Amp = -15.
# Red noise power law spectral index 
# [1, 7]
gam = 13./3.

In [18]:
# Load environment from .env file
load_dotenv()

True

In [20]:
# Random number seed
NP_SEED = int(os.getenv("NP_SEED"))
if NP_SEED is not None:
    np.random.seed(NP_SEED)

# 15-year dataset
DATA_15YR_ROOT = Path(os.getenv("DATA_15YR_ROOT"))
DATA_15YR_PAR = DATA_15YR_ROOT / "narrowband/par"
DATA_15YR_TIM = DATA_15YR_ROOT / "narrowband/tim"

In [25]:
PSR_DICT = {
    "B1937+21": {
        "RA": 294.91067,
        "DEC": 21.58309,
    },
    "J1312+0051": {
        "RA": 198.19434,
        "DEC": 0.85003,
    },
}
N_PSR = len(PSR_DICT.keys())

In [26]:
for pname in PSR_DICT.keys():
    par_path = [p for p in DATA_15YR_PAR.glob(f"{pname}_PINT*.par")]
    tim_path = [p for p in DATA_15YR_TIM.glob(f"{pname}_PINT*.tim")]
    PSR_DICT[pname]["par"] = par_path[0]
    PSR_DICT[pname]["tim"] = tim_path[0]

In [28]:
# Zero residuals
for pname in PSR_DICT.keys():
    print(f"Zeroing residuals for {pname}")
    model, toas = get_model_and_toas(PSR_DICT[pname]["par"], PSR_DICT[pname]["tim"])
    zero_residuals(toas, model)
    toas.print_summary()
    PSR_DICT[pname]["model"] = model
    PSR_DICT[pname]["toas"] = toas

Zeroing residuals for B1937+21
Number of TOAs:  23023
Paradigm: narrowband
Number of commands:  1
Number of observatories: 2 ['gbt', 'arecibo']
MJD span:  53267.086 to 59065.129
Date span: 2004-09-19 02:03:50.827975157 to 2020-08-04 03:05:49.822749800
arecibo TOAs (7180):
  Min freq:      1151.875 MHz
  Max freq:      2397.702 MHz
  Min error:     0.006 us
  Max error:     4.2 us
  Median error:  0.053 us
gbt TOAs (15843):
  Min freq:      724.686 MHz
  Max freq:      1887.500 MHz
  Min error:     0.014 us
  Max error:     3.38 us
  Median error:  0.058 us

Zeroing residuals for J1312+0051
Number of TOAs:  1705
Paradigm: narrowband
Number of commands:  1
Number of observatories: 1 ['arecibo']
MJD span:  57389.438 to 59055.891
Date span: 2016-01-02 10:31:05.009320227 to 2020-07-25 21:23:21.299437184
arecibo TOAs (1705):
  Min freq:      1154.382 MHz
  Max freq:      2397.920 MHz
  Min error:     0.741 us
  Max error:     51.3 us
  Median error:  7.27 us



In [30]:
# Get MJDs
for pname in PSR_DICT.keys():
    print(f"Getting MJDs for {pname}")
    PSR_DICT[pname]["mjds"] = PSR_DICT[pname]["toas"].get_mjds()

Getting MJDs for B1937+21
Getting MJDs for J1312+0051


In [31]:
mjd_start = None
mjd_end = None

for pname in PSR_DICT.keys():
    min_mjd = np.min(PSR_DICT[pname]["mjds"]).value
    max_mjd = np.max(PSR_DICT[pname]["mjds"]).value

    if mjd_start is None:
        mjd_start = min_mjd
    elif min_mjd < mjd_start:
        mjd_start = min_mjd

    if mjd_end is None:
        mjd_end = max_mjd
    elif max_mjd > mjd_end:
        mjd_end = max_mjd

In [33]:
# Number of days to add on either side
DAY_PAD = 1
# Number of points
N_POINTS = 600

# gw start and end times for entire data set
t_start = (mjd_start - DAY_PAD) * 86400
t_stop  = (mjd_end + DAY_PAD) * 86400

# duration of the signal
dur = t_stop - t_start

In [34]:
# make a vector of evenly sampled data points
ut = np.linspace(t_start, t_stop, N_POINTS)
# time resolution in days
dt = dur / N_POINTS

In [ ]:
# pulsar sky locations
psrlocs = np.zeros((N_PSR, 2))

In [ ]:
# compute the overlap reduction function

psrlocs = np.zeros((Npulsars, 2))

for ii in range(Npulsars):
    if "RAJ" and "DECJ" in psr[ii].pars():
        psrlocs[ii] = np.double(psr[ii]["RAJ"].val), np.double(psr[ii]["DECJ"].val)
    elif "ELONG" and "ELAT" in psr[ii].pars():
        fac = 180.0 / np.pi
        # check for B name
        if "B" in psr[ii].name:
            epoch = "1950"
        else:
            epoch = "2000"
        coords = ephem.Equatorial(
            ephem.Ecliptic(str(psr[ii]["ELONG"].val * fac), str(psr[ii]["ELAT"].val * fac)), epoch=epoch
        )
        psrlocs[ii] = float(repr(coords.ra)), float(repr(coords.dec))

psrlocs[:, 1] = np.pi / 2.0 - psrlocs[:, 1]
anisbasis = np.array(anis.CorrBasis(psrlocs, lmax))
ORF = sum(clm[kk] * anisbasis[kk] for kk in range(len(anisbasis)))
ORF *= 2.0